# TP 3 — Nettoyer et tester : un module de transformations PySpark
**Big Data Engineering — Master 1 — DMI/FST/UCAD — Prof. Samba Ndiaye**

Objectif : transformer `customers.csv` (sale) en une table clients **propre**,
avec un code **modulaire et testé**.

**Consignes**
- Complétez chaque cellule marquée `# === À COMPLÉTER ===` (remplacez les `...`).
- Exécutez le notebook **de bout en bout** sans erreur.
- Poussez le notebook **avec ses sorties** sur votre dépôt GitHub.

Rappel : les fonctions de nettoyage « réelles » vivent dans `src/transformations.py`.
Ce notebook **démontre** et **mesure** ; il importe le module.


## 0. Vérification de l'environnement


In [1]:
import sys
import pyspark
from pyspark.sql import SparkSession, functions as F

print("Python  :", sys.version.split()[0])
print("PySpark :", pyspark.__version__)

spark = (SparkSession.builder
         .master("local[*]")
         .appName("TP3-nettoyage")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
spark


Python  : 3.10.11
PySpark : 3.5.1


### Tableau de relevés
On consigne ici les mesures au fil du TP (à reporter dans `docs/QUALITE.md`).


In [2]:
releves = {
    "lignes_brutes": None,
    "emails_manquants": None,
    "villes_distinctes_avant": None,
    "villes_distinctes_apres": None,
    "doublons_exacts": None,
    "lignes_apres_nettoyage": None,
}
releves


{'lignes_brutes': None,
 'emails_manquants': None,
 'villes_distinctes_avant': None,
 'villes_distinctes_apres': None,
 'doublons_exacts': None,
 'lignes_apres_nettoyage': None}

## 1. Charger avec un schéma explicite
On impose le schéma plutôt que de le laisser deviner (fiabilité + vitesse).


In [3]:
from pyspark.sql.types import StructType, StructField, StringType

schema_clients = StructType([
    StructField("customer_id",     StringType(), False),
    StructField("prenom",          StringType(), True),
    StructField("nom",             StringType(), True),
    StructField("email",           StringType(), True),
    StructField("telephone",       StringType(), True),
    StructField("ville",           StringType(), True),
    StructField("region",          StringType(), True),
    StructField("date_naissance",  StringType(), True),
    StructField("date_inscription",StringType(), True),
])

df_brut = (spark.read.option("header", True)
                 .schema(schema_clients)
                 .csv("../data/customers.csv"))

releves["lignes_brutes"] = df_brut.count()
df_brut.printSchema()
print("lignes :", releves["lignes_brutes"])


root
 |-- customer_id: string (nullable = true)
 |-- prenom: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- email: string (nullable = true)
 |-- telephone: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- region: string (nullable = true)
 |-- date_naissance: string (nullable = true)
 |-- date_inscription: string (nullable = true)

lignes : 5025


## 2. Diagnostic : mesurer les défauts
On **mesure** chaque défaut avant de corriger quoi que ce soit.

### 2.1 Valeurs manquantes par colonne


In [4]:
# CORRIGÉ
df_brut.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_brut.columns
]).show()

+-----------+------+---+-----+---------+-----+------+--------------+----------------+
|customer_id|prenom|nom|email|telephone|ville|region|date_naissance|date_inscription|
+-----------+------+---+-----+---------+-----+------+--------------+----------------+
|          0|     0|  0|   75|        0|    0|     0|             0|               0|
+-----------+------+---+-----+---------+-----+------+--------------+----------------+



### 2.2 Faux manquants (emails "" ou "N/A")


In [5]:
# CORRIGÉ
nb_email_vide = df_brut.filter(
    (F.trim(F.col("email")) == "") | (F.col("email") == "N/A")
    | F.col("email").isNull()
).count()
print("emails vides, N/A ou null :", nb_email_vide)
releves["emails_manquants"] = nb_email_vide
# Observation attendue : de l'ordre de ~3 % des clients.

emails vides, N/A ou null : 150


### 2.3 Villes distinctes (avant normalisation) et doublons exacts


In [6]:
# CORRIGÉ
releves["villes_distinctes_avant"] = df_brut.select("ville").distinct().count()
releves["doublons_exacts"] = df_brut.count() - df_brut.distinct().count()
print(releves["villes_distinctes_avant"], "villes distinctes (brut)")
print(releves["doublons_exacts"], "doublons exacts")
# Observation attendue : bien plus de 19 villes a cause de la casse/accents.

499 villes distinctes (brut)
15 doublons exacts


## 3. Les fonctions de transformation (dans src/)
En production, ces fonctions sont dans `src/transformations.py` et **testées**.
Ici on les définit dans le notebook pour la démonstration, **à l'identique**.

> Dans votre livrable, déplacez-les dans `src/transformations.py` et importez-les.

### 3.1 Manquants et email


In [7]:
from pyspark.sql import DataFrame

def unifier_manquants(df: DataFrame) -> DataFrame:
    """Emails "" / "N/A" -> null."""
    e = F.trim(F.col("email"))
    return df.withColumn(
        "email",
        F.when(e.isin("", "N/A", "n/a", "NULL"), None).otherwise(e))

def normaliser_email(df: DataFrame) -> DataFrame:
    """Email en minuscules + trim ; drapeau de validite."""
    motif = r"^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$"
    df = df.withColumn("email", F.lower(F.trim(F.col("email"))))
    return df.withColumn(
        "email_valide",
        F.when(F.col("email").isNull(), F.lit(None))
         .otherwise(F.col("email").rlike(motif)))

### 3.2 Ville (avec retrait d'accents)


In [8]:
import unicodedata
from pyspark.sql.types import StringType

def sans_accent(s):
    if s is None:
        return None
    nfkd = unicodedata.normalize("NFKD", s)
    return "".join(c for c in nfkd if not unicodedata.combining(c))

sans_accent_udf = F.udf(sans_accent, StringType())

def normaliser_ville(df: DataFrame) -> DataFrame:
    """ville (affichage) + ville_norm (cle sans accent)."""
    df = df.withColumn("ville", F.initcap(F.trim(F.col("ville"))))
    return df.withColumn(
        "ville_norm",
        F.lower(sans_accent_udf(F.trim(F.col("ville")))))

### 3.3 Téléphone et date de naissance


In [9]:
def normaliser_telephone(df: DataFrame) -> DataFrame:
    """9 chiffres, prefixe 70/75/76/77/78 ; drapeau de validite."""
    tel = F.regexp_replace(F.col("telephone"), r"[^0-9]", "")
    tel = F.regexp_replace(tel, r"^221", "")
    return (df.withColumn("tel_norm", tel)
              .withColumn("tel_valide",
                  tel.rlike(r"^(70|75|76|77|78)\d{7}$")))

def valider_naissance(df: DataFrame) -> DataFrame:
    """Date plausible entre 1920 et aujourd'hui, sinon null."""
    d = F.to_date(F.col("date_naissance"), "yyyy-MM-dd")
    return df.withColumn(
        "date_naissance",
        F.when((d >= F.lit("1920-01-01")) & (d <= F.current_date()), d)
         .otherwise(None))

### 3.4 Déduplication (après normalisation)


In [10]:
def dedupliquer_clients(df: DataFrame) -> DataFrame:
    """Doublons exacts puis 1 ligne par customer_id."""
    from pyspark.sql.window import Window
    w = Window.partitionBy("customer_id").orderBy(
        F.col("date_inscription").desc())
    return (df.dropDuplicates()
              .withColumn("_r", F.row_number().over(w))
              .filter(F.col("_r") == 1)
              .drop("_r"))

## 4. Assembler le pipeline et mesurer l'effet


In [11]:
def nettoyer_clients(df: DataFrame) -> DataFrame:
    return (df
        .transform(unifier_manquants)
        .transform(normaliser_email)
        .transform(normaliser_ville)
        .transform(normaliser_telephone)
        .transform(valider_naissance)
        .transform(dedupliquer_clients))

df_net = nettoyer_clients(df_brut)

releves["villes_distinctes_apres"] = df_net.select("ville_norm").distinct().count()
releves["lignes_apres_nettoyage"]  = df_net.count()
print("avant :", releves["lignes_brutes"], "-> apres :", releves["lignes_apres_nettoyage"])
print("villes distinctes :", releves["villes_distinctes_avant"],
      "->", releves["villes_distinctes_apres"])


avant : 5025 -> apres : 5000
villes distinctes : 499 -> 499


### 4.1 Vérification visuelle : top des villes après nettoyage


In [12]:
df_net.groupBy("ville_norm").count().orderBy(F.desc("count")).show(10)


+--------------------+-----+
|          ville_norm|count|
+--------------------+-----+
|           rue gomes|   20|
|    97, avenue robin|   19|
|71, avenue mathil...|   19|
|     55, rue laurent|   18|
|936, boulevard de...|   18|
|      561, rue perez|   18|
| 53, boulevard louis|   17|
|  avenue david faure|   17|
|  1, chemin valentin|   17|
|309, avenue de le...|   17|
+--------------------+-----+
only showing top 10 rows



### 4.2 Tableau de relevés final


In [13]:
for k, v in releves.items():
    print(f"{k:30s} : {v}")


lignes_brutes                  : 5025
emails_manquants               : 150
villes_distinctes_avant        : 499
villes_distinctes_apres        : 499
doublons_exacts                : 15
lignes_apres_nettoyage         : 5000


## 5. Questions de réflexion
Répondez en quelques lignes (cellule markdown ci-dessous).

1. Combien de villes distinctes **avant** et **après** normalisation ? Que
   conclure sur l'effet de la casse et des accents ?
2. Quelle décision avez-vous prise pour les emails manquants (drop ou fill) ?
   Pourquoi ?
3. Vous avez écrit une **UDF** (`sans_accent`). À quel coût ? Pourquoi est-elle
   justifiée ici alors que la règle est « fonctions intégrées d'abord » ?
4. En quoi la déduplication **après** normalisation diffère-t-elle d'une
   déduplication naïve ?


*Votre réponse :*

1. Avant normalisation, il y avait **499 villes distinctes** et après normalisation également **499 villes distinctes**. Cela indique que, dans ce jeu de données, la normalisation de la casse et la suppression des accents n'ont pas réduit le nombre de valeurs uniques.

2. Nous avons choisi de conserver les lignes contenant des emails manquants (**fill/conservation**) après avoir uniformisé les différentes formes de valeurs manquantes en NULL. Cette décision permet de ne pas perdre des observations utiles pour les autres attributs. Les emails absents restent identifiés comme des valeurs manquantes et ne sont pas considérés comme valides.

3. L'utilisation d'une UDF entraîne un coût en performance car Spark doit exécuter du code Python ligne par ligne, ce qui empêche certaines optimisations du moteur Spark et ajoute un coût de sérialisation entre la JVM et Python.

4. Une déduplication naïve compare directement les données brutes. Elle peut donc considérer comme différentes des valeurs qui représentent pourtant la même information, par exemple "DAKAR", "Dakar" ou "dàkar".
Ici, la normalisation est effectuée avant la déduplication : les valeurs sont harmonisées (suppression des espaces, uniformisation de la casse et suppression des accents) afin de faciliter l'identification des doublons.

## 6. Vers le livrable
1. Déplacez les fonctions de la section 3 dans `src/transformations.py`.
2. Écrivez les tests dans `tests/test_transformations.py` (+ `conftest.py`).
3. Vérifiez `pytest -q` : **tout au vert**.
4. Remplissez `docs/QUALITE.md` avec le tableau de relevés.
5. Poussez le tout :

```bash
git add src/ tests/ notebooks/ docs/
git commit -m "feat: module de nettoyage clients + tests (TP3)"
git push
```

> Rappel : `data/` n'est **jamais** commité.


In [19]:
df_brut.count() - df_brut.distinct().count()

15

In [ ]:
# Arret propre de la session Spark
spark.stop()
print("Session fermee. Notebook termine.")
